# 19_bronze_ddl_avro_schema

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName("olist-bronze-setup")
        .getOrCreate()
)

# Load the Spark SQL magic python file created earlier
from spark_sql_magic import sql  # registers the %%sql magic automatically

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Config

In [2]:
INGESTION_CONFIG = [
    "customers",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers"
]

CATALOG = "polaris"
NAMESPACE = "bronze"
KAFKA_BOOTSTRAP = "kafka:9092"

## Function: Fetch Schema

In [3]:
import requests

def fetch_schema(topic):
    url = f"http://apicurio:8081/apis/registry/v2/groups/default/artifacts/olist.oltp.{topic}-value/versions/latest"
    response = requests.get(url, timeout = 10)
    response.raise_for_status()
    return response.json()

## Function: Create Bronze Table

In [4]:
from pyspark.sql.avro.functions import from_avro

def create_bronze_table(topic):
    table_name = f"{CATALOG}.{NAMESPACE}.{topic}"
    
    if spark.catalog.tableExists(table_name):
        print(f"Table {table_name} exists. Skipping.")
        return

    print(f"Creating Bronze table {table_name}")

    # 1. Fetch Schema
    schema_dict = fetch_schema(topic)
    schema_json = json.dumps(schema_dict)

    # 2. Create empty Kafka DF just for schema extraction
    df_schema = (
        spark.createDataFrame([], "value BINARY")
        .select(from_avro("value", schema_json).alias("data"))
        .select("data.*")
    )

    # 3. Parse Avro
    df_final = (
        df_schema
            .withColumn("offset", F.lit(None).cast("long"))
            .withColumn("partition", F.lit(None).cast("int"))
            .withColumn("kafka_ingest_ts", F.lit(None).cast("timestamp"))
            .withColumnRenamed("created_at", "pg_created_at")
            .withColumnRenamed("updated_at", "pg_updated_at")
            .withColumnRenamed("__op", "cdc_op")
            .withColumnRenamed("__ts_ms", "pg_commit_ts")
            .withColumn("batch_id", F.lit(None).cast("string"))
            .withColumn("spark_ingest_ts", F.current_timestamp())
    )

    # 4. Create table
    (
        df_final
        .writeTo(table_name)
        .tableProperty("format-version", "2")  # important for CDC
        .create()
    )

    print(f"Created {table_name}")

In [5]:
for topic in INGESTION_CONFIG:
    create_bronze_table(topic)

Creating Bronze table polaris.bronze.customers


Created polaris.bronze.customers
Creating Bronze table polaris.bronze.order_items
Created polaris.bronze.order_items
Creating Bronze table polaris.bronze.order_payments


Created polaris.bronze.order_payments
Creating Bronze table polaris.bronze.order_reviews
Created polaris.bronze.order_reviews
Creating Bronze table polaris.bronze.orders
Created polaris.bronze.orders
Creating Bronze table polaris.bronze.products


Created polaris.bronze.products
Creating Bronze table polaris.bronze.sellers
Created polaris.bronze.sellers


## Sanity Check All Tables

In [6]:
%%sql
SHOW TABLES IN bronze;

+---------+--------------+-----------+
|namespace|tableName     |isTemporary|
+---------+--------------+-----------+
|bronze   |products      |false      |
|bronze   |order_payments|false      |
|bronze   |orders        |false      |
|bronze   |sellers       |false      |
|bronze   |customers     |false      |
|bronze   |order_items   |false      |
|bronze   |order_reviews |false      |
+---------+--------------+-----------+



In [7]:
for item in INGESTION_CONFIG:
    df = spark.table(f"polaris.bronze.{item}")
    df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- pg_created_at: long (nullable = true)
 |-- pg_updated_at: long (nullable = true)
 |-- cdc_op: string (nullable = true)
 |-- pg_commit_ts: long (nullable = true)
 |-- offset: long (nullable = true)
 |-- partition: integer (nullable = true)
 |-- kafka_ingest_ts: timestamp (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- spark_ingest_ts: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: long (nullable = true)
 |-- price: struct (nullable = true)
 |    |-- scale: integer (nullable = true)
 |    |-- value: binary (nullable = true)
 |-- freight_value: struc

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  